# Statistical testing

Real hypothesis tests for a few surface-level tweet properties, before
reaching for any text vectorization.

In [1]:
import pandas as pd
from scipy import stats

train = pd.read_pickle('data/train_sample.pkl')
train.shape

(20000, 7)

### Does tweet length differ by sentiment?

In [2]:
u_stat, p_length = stats.mannwhitneyu(train[train.label == 1].length, train[train.label == 0].length)
print(f'Mann-Whitney U = {u_stat:.0f}, p = {p_length:.3f}')
train.groupby('label').length.mean()

Mann-Whitney U = 49358278, p = 0.116


label
0    74.5774
1    73.6778
Name: length, dtype: float64

Not significant (p = 0.12). Positive and negative tweets are, on average, about the same length, length alone tells you nothing about sentiment here.

### Does mentioning someone (@) relate to sentiment?

In [3]:
has_mention = train.text.str.contains('@')
chi2_mention, p_mention, dof, exp = stats.chi2_contingency(pd.crosstab(train.label, has_mention))
print(f'chi2 = {chi2_mention:.1f}, p = {p_mention:.2e}')
pd.crosstab(train.label, has_mention, normalize = 'index')

chi2 = 477.2, p = 8.54e-106


text,False,True
label,,
0,0.6118,0.3882
1,0.4576,0.5424


Highly significant. Positive tweets mention someone 54% of the time versus 39% for negative tweets, positive tweets skew more conversational/social (replies, thanks, shoutouts), negative tweets more often just vent without directing it at anyone.

### Does using an exclamation mark relate to sentiment?

In [4]:
has_exclaim = train.text.str.contains('!')
chi2_exclaim, p_exclaim, dof, exp = stats.chi2_contingency(pd.crosstab(train.label, has_exclaim))
print(f'chi2 = {chi2_exclaim:.1f}, p = {p_exclaim:.2e}')
pd.crosstab(train.label, has_exclaim, normalize = 'index')

chi2 = 219.8, p = 9.78e-50


text,False,True
label,,
0,0.7459,0.2541
1,0.6495,0.3505


Also significant, positive tweets use exclamation marks 35% of the time versus 25% for negative, matching the intuition that enthusiasm shows up in punctuation.

In [5]:
import json, os
os.makedirs('outputs', exist_ok = True)
with open('outputs/statistical_tests.json', 'w') as f:
    json.dump({
        'length_mannwhitney_p': float(p_length),
        'mention_chi2_p': float(p_mention),
        'exclaim_chi2_p': float(p_exclaim),
    }, f, indent = 2)